# 02 ET PseudoGT DQA Probe 2h

このノートブックは、DQAに入れる前のEfficientTeacher追加学習が本当に有効な更新になっているかを短時間で確認するための検証です。前回の結果ではpseudoGTが多すぎる・クラスが偏る・bbox/cls lossまで強く入れて教師の誤検出を学習してしまう、という疑いが強かったので、ここではETのloss重みとpseudoGTの作り方を絞って比較します。

見たいことは3つです。まずpseudoGT密度がGT密度に近づくか。次に各client local checkpointがwarmupより壊れていないか。最後にFedAvgではなくDQA集約でmAPが伸びる条件があるかです。

In [1]:
print("Hello, World!")

Hello, World!


In [2]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    markers = (
        "efficient_teacher/et_pseudogt_dqa_probe_02.py",
        "dynamic_quality_aware_classwise_aggregation/exploring/dqa_probe_02_2.py",
        "navigating_data_heterogeneity/vendor/efficientteacher/train.py",
    )
    for base in (start, *start.parents):
        candidates = (base, base / "Object_Detection")
        for candidate in candidates:
            if all((candidate / marker).exists() for marker in markers):
                return candidate
    raise FileNotFoundError("Object_Detection repository root was not found.")


REPO_ROOT = find_repo_root()
ET_ROOT = REPO_ROOT / "efficient_teacher"
if str(ET_ROOT) not in sys.path:
    sys.path.insert(0, str(ET_ROOT))

from et_pseudogt_dqa_probe_02 import (  # noqa: E402
    ETPseudoGTDQAProbe,
    ETPseudoGTProbeConfig,
    reset_probe_outputs,
)

pd.options.display.max_columns = 200
pd.options.display.max_rows = 80
print("repo:", REPO_ROOT)

repo: /app/Object_Detection


## 1. 実行設定

`RUN_FULL_PROBE=True`で一括実行します。既存結果を再利用したいときは`FORCE_RERUN=False`のままで大丈夫です。完全に作り直す場合だけ`RESET_OUTPUTS=True`にしてください。

In [3]:
RUN_FULL_PROBE = True
FORCE_RERUN = False
RESET_OUTPUTS = False
INCLUDE_LABELMATCH = False

cfg = ETPseudoGTProbeConfig(
    source_warmup_key="dqa_v2_scene_12h",
    experiment_name="efficientteacher_pseudogt_dqa_probe_2h",
    server_train_limit=1024,
    server_val_limit=512,
    client_target_limit=512,
    max_wall_hours=2.0,
    batch_size=8,
    val_batch_size=4,
    force_rerun=FORCE_RERUN,
    run_client_local_eval=True,
    run_server_calibration=True,
    top_calibration_k=4,
)

probe = ETPseudoGTDQAProbe(cfg, include_labelmatch=INCLUDE_LABELMATCH)
if RESET_OUTPUTS:
    reset_probe_outputs(probe)
    probe = ETPseudoGTDQAProbe(cfg, include_labelmatch=INCLUDE_LABELMATCH)

display(probe.describe())

{'experiment_root': '/app/Object_Detection/efficient_teacher/efficientteacher_pseudogt_dqa_probe_2h',
 'warmup': '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa_ver2_scene_12h/global_checkpoints/round000_warmup.pt',
 'client_variants': ['et_default_lr1e-3',
  'et_reliable_only_strict',
  'et_capped_balanced_obj',
  'et_high_precision_capped'],
 'aggregation_variants': ['fedavg', 'dqa_v2_default', 'dqa_v2_conservative'],
 'calibration_variants': ['cal_all_lr1e-3_ep1'],
 'budget_hours': 2.0,
 'purpose': 'ET pseudo-label and unsupervised-loss probe for DQA input quality',
 'client_target_limit': 512,
 'server_val_limit': 512,
 'top_calibration_k': 4}

## 2. データ確認

server側GTとclient側targetの画像数・物体数を先に確認します。pseudoGT密度はこのserver validation GT密度と比べます。

In [4]:
audit = probe.prepare_data()
cols = [c for c in ["role", "client_id", "weather", "images", "objects", "objects_per_image", "has_gt"] if c in audit.columns]
display(audit[cols])

,role,weather,images,objects,has_gt
0,server_train,partly_cloudy,4881,97123.0,True
1,server_val,partly_cloudy,738,14937.0,True
2,mini_server_train,partly_cloudy,1024,20881.0,True
3,mini_server_val,partly_cloudy,512,10082.0,True
4,client_0_target_full,overcast,5000,NaN,False
5,client_0_target_mini,overcast,512,NaN,False
6,client_1_target_full,rainy,5000,NaN,False
7,client_1_target_mini,rainy,512,NaN,False
8,client_2_target_full,snowy,5000,NaN,False
9,client_2_target_mini,snowy,512,NaN,False


## 3. 比較するET設定

`et_default_lr1e-3`を今のET/FedSTO寄りの基準にして、残りはpseudoGTを厳しくする・bbox/clsの影響を弱める・画像あたり/クラスあたりのpseudoGT数を制限する、という方向で比較します。

In [5]:
variant_rows = []
for item in probe.client_variants:
    ssod = item.get("ssod_overrides", {})
    env = item.get("extra_env", {})
    variant_rows.append(
        {
            "variant": item["name"],
            "epochs": item["epochs"],
            "lr0": item["lr0"],
            "train_scope": item["train_scope"],
            "nms_conf": ssod.get("nms_conf_thres", "default"),
            "low": ssod.get("ignore_thres_low", "default"),
            "high": ssod.get("ignore_thres_high", "default"),
            "teacher_w": ssod.get("teacher_loss_weight", "default"),
            "box_w": ssod.get("box_loss_weight", "default"),
            "obj_w": ssod.get("obj_loss_weight", "default"),
            "cls_w": ssod.get("cls_loss_weight", "default"),
            "uc_bbox": ssod.get("pseudo_label_with_bbox", "default"),
            "uc_cls": ssod.get("pseudo_label_with_cls", "default"),
            "cap_img": env.get("ET_MAX_PSEUDO_PER_IMAGE", ""),
            "cap_cls_img": env.get("ET_MAX_PSEUDO_PER_CLASS_IMAGE", ""),
            "note": item.get("note", ""),
        }
    )
display(pd.DataFrame(variant_rows))

,variant,epochs,lr0,train_scope,nms_conf,low,high,teacher_w,box_w,obj_w,cls_w,uc_bbox,uc_cls,cap_img,cap_cls_img,note
0,et_default_lr1e-3,1,0.0010,all,default,default,default,default,default,default,default,default,default,,,Current EfficientTeacher/FedSTO pseudo-label d...
1,et_reliable_only_strict,1,0.0003,all,0.35,0.35,0.75,0.35,0.02,0.35,0.1,False,False,,,Strict reliable labels; uncertain labels affec...
2,et_capped_balanced_obj,1,0.0003,all,0.25,0.3,0.72,0.3,0.015,0.3,0.05,False,False,28,8,Cap pseudo boxes near GT density; keep uncerta...
3,et_high_precision_capped,1,0.0003,neck_head,0.5,0.5,0.85,0.25,0.015,0.25,0.05,False,False,24,6,High-confidence capped pseudo labels; update n...


## 4. 2時間プローブ実行

実行内容は、warmup評価、clientごとのET追加学習、pseudoGT集計、client local評価、FedAvg/DQA集約評価、上位候補だけのserver GT 1 epoch補正です。残り時間が少なくなると重い処理は自動でskipされます。

In [6]:
if RUN_FULL_PROBE:
    outputs = probe.run_all()
    print("done")
else:
    outputs = {}
    print("RUN_FULL_PROBE is False; reading existing results only.")

done


## 5. 結果テーブル

主要CSVを読み出します。`recommendation_table.csv`が最終判断用で、pseudoGT密度、client local、FedAvg、DQA、server補正後の結果をvariant単位でまとめています。

In [7]:
RESULT_ROOT = probe.result_root


def read_result(filename: str) -> pd.DataFrame:
    path = RESULT_ROOT / filename
    if not path.exists():
        print(f"missing: {path}")
        return pd.DataFrame()
    df = pd.read_csv(path)
    print(f"loaded: {filename} ({len(df)} rows)")
    return df


pseudo_density = read_result("pseudo_density_summary.csv")
client_eval = read_result("client_local_eval_summary.csv")
aggregation = read_result("aggregation_summary.csv")
calibration = read_result("server_calibration_summary.csv")
leaderboard = read_result("overall_leaderboard.csv")
recommendation = read_result("recommendation_table.csv")

if len(pseudo_density):
    density_cols = [
        "variant",
        "weather",
        "pseudo_total",
        "pseudo_per_image",
        "reference_gt_per_image",
        "pseudo_to_gt_density",
        "top_class_share",
        "active_classes",
        "mean_quality_active",
    ]
    display(pseudo_density[[c for c in density_cols if c in pseudo_density.columns]].sort_values(["variant", "weather"]))

if len(aggregation):
    agg_cols = ["variant", "aggregation", "status", "map50_final", "map50_95_final", "delta_map50_95_vs_warmup"]
    display(aggregation[[c for c in agg_cols if c in aggregation.columns]].sort_values("map50_95_final", ascending=False))

if len(recommendation):
    display(recommendation)

loaded: pseudo_density_summary.csv (12 rows)
loaded: client_local_eval_summary.csv (12 rows)
loaded: aggregation_summary.csv (12 rows)
loaded: server_calibration_summary.csv (4 rows)
loaded: overall_leaderboard.csv (30 rows)
loaded: recommendation_table.csv (4 rows)


,variant,weather,pseudo_total,pseudo_per_image,reference_gt_per_image,pseudo_to_gt_density,top_class_share,active_classes,mean_quality_active
6,et_capped_balanced_obj,overcast,8404.0,16.414062,19.691406,0.833565,0.426702,8,0.665892
7,et_capped_balanced_obj,rainy,7989.0,15.603516,19.691406,0.792402,0.442358,8,0.658451
8,et_capped_balanced_obj,snowy,8389.0,16.384766,19.691406,0.832077,0.422458,7,0.674613
0,et_default_lr1e-3,overcast,32785.0,64.033203,19.691406,3.251835,0.679488,9,0.495625
1,et_default_lr1e-3,rainy,34686.0,67.746094,19.691406,3.440389,0.724269,9,0.487741
2,et_default_lr1e-3,snowy,34626.0,67.628906,19.691406,3.434438,0.693438,8,0.497955
9,et_high_precision_capped,overcast,5036.0,9.835938,19.691406,0.499504,0.539317,7,0.800160
10,et_high_precision_capped,rainy,4653.0,9.087891,19.691406,0.461516,0.573608,8,0.764648
11,et_high_precision_capped,snowy,4801.0,9.376953,19.691406,0.476195,0.555301,7,0.783644
3,et_reliable_only_strict,overcast,15278.0,29.839844,19.691406,1.515374,0.743291,8,0.691350


,variant,aggregation,status,map50_final,map50_95_final
0,et_default_lr1e-3,fedavg,ok,0.453,0.26
1,et_default_lr1e-3,dqa_v2_default,ok,0.453,0.26
2,et_default_lr1e-3,dqa_v2_conservative,ok,0.453,0.26
3,et_reliable_only_strict,fedavg,ok,0.453,0.26
4,et_reliable_only_strict,dqa_v2_default,ok,0.453,0.26
5,et_reliable_only_strict,dqa_v2_conservative,ok,0.453,0.26
6,et_capped_balanced_obj,fedavg,ok,0.453,0.26
7,et_capped_balanced_obj,dqa_v2_default,ok,0.453,0.26
8,et_capped_balanced_obj,dqa_v2_conservative,ok,0.453,0.26
9,et_high_precision_capped,fedavg,ok,0.453,0.26


,variant,pseudo_per_image_mean,pseudo_per_image_max,pseudo_to_gt_density_mean,top_class_share_mean,...,dqa_map50_95_best,dqa_map50_best,calibration_map50_95_max,calibration_map50_max,delta_dqa_vs_fedavg_map50_95
0,et_capped_balanced_obj,16.134115,16.414062,0.819348,0.430506,...,0.26,0.453,NaN,NaN,0.0
1,et_default_lr1e-3,66.469401,67.746094,3.375554,0.699065,...,0.26,0.453,0.26725,0.46689,0.0
2,et_high_precision_capped,9.433594,9.835938,0.479072,0.556075,...,0.26,0.453,NaN,NaN,0.0
3,et_reliable_only_strict,26.134115,29.839844,1.327184,0.720790,...,0.26,0.453,0.26722,0.46673,0.0


## 6. 判定

DQAに渡す価値がある候補かを機械的に判定します。目安は、DQAがFedAvg以上、pseudoGT密度がGTの2倍以内、最大クラス占有率が高すぎないことです。

In [8]:
baseline = None
if len(leaderboard) and "name" in leaderboard.columns:
    warmup = leaderboard.loc[leaderboard["name"].eq("warmup_same_mini"), "map50_95_final"].dropna()
    if len(warmup):
        baseline = float(warmup.iloc[0])

if len(recommendation):
    decision = recommendation.copy()
    if baseline is not None and "aggregate_map50_95_max" in decision.columns:
        decision["delta_best_aggregate_vs_warmup"] = decision["aggregate_map50_95_max"] - baseline
    needed = {"delta_dqa_vs_fedavg_map50_95", "pseudo_to_gt_density_mean", "top_class_share_mean"}
    if needed.issubset(decision.columns):
        decision["dqa_candidate"] = (
            (decision["delta_dqa_vs_fedavg_map50_95"].fillna(-999) >= -0.0005)
            & (decision["pseudo_to_gt_density_mean"].fillna(999) <= 2.0)
            & (decision["top_class_share_mean"].fillna(1.0) <= 0.85)
        )
    sort_cols = [c for c in ["dqa_candidate", "delta_best_aggregate_vs_warmup", "aggregate_map50_95_max"] if c in decision.columns]
    if sort_cols:
        display(decision.sort_values(sort_cols, ascending=[False] * len(sort_cols)))
    else:
        display(decision)
    print("warmup map50_95:", baseline)
else:
    print("recommendation_table.csv is not available yet.")

,variant,pseudo_per_image_mean,pseudo_per_image_max,pseudo_to_gt_density_mean,top_class_share_mean,...,calibration_map50_95_max,calibration_map50_max,delta_dqa_vs_fedavg_map50_95,delta_best_aggregate_vs_warmup,dqa_candidate
0,et_capped_balanced_obj,16.134115,16.414062,0.819348,0.430506,...,NaN,NaN,0.0,0.0,True
2,et_high_precision_capped,9.433594,9.835938,0.479072,0.556075,...,NaN,NaN,0.0,0.0,True
3,et_reliable_only_strict,26.134115,29.839844,1.327184,0.720790,...,0.26722,0.46673,0.0,0.0,True
1,et_default_lr1e-3,66.469401,67.746094,3.375554,0.699065,...,0.26725,0.46689,0.0,0.0,False


warmup map50_95: 0.26


## 7. 出力先

結果は下記に保存されます。追加考察では、`pseudo_density_summary.csv`でpseudoGTの量と偏りを見て、`aggregation_summary.csv`でFedAvg/DQA差分を見て、最後に`recommendation_table.csv`で設定単位の結論を出します。

In [9]:
paths = {
    "experiment_root": probe.exp_root,
    "results": probe.result_root,
    "pseudo_density": probe.result_root / "pseudo_density_summary.csv",
    "client_local_eval": probe.result_root / "client_local_eval_summary.csv",
    "aggregation": probe.result_root / "aggregation_summary.csv",
    "server_calibration": probe.result_root / "server_calibration_summary.csv",
    "recommendation": probe.result_root / "recommendation_table.csv",
    "leaderboard": probe.result_root / "overall_leaderboard.csv",
}
for key, path in paths.items():
    print(f"{key}: {path}")

experiment_root: /app/Object_Detection/efficient_teacher/efficientteacher_pseudogt_dqa_probe_2h
results: /app/Object_Detection/efficient_teacher/efficientteacher_pseudogt_dqa_probe_2h/results
pseudo_density: /app/Object_Detection/efficient_teacher/efficientteacher_pseudogt_dqa_probe_2h/results/pseudo_density_summary.csv
client_local_eval: /app/Object_Detection/efficient_teacher/efficientteacher_pseudogt_dqa_probe_2h/results/client_local_eval_summary.csv
aggregation: /app/Object_Detection/efficient_teacher/efficientteacher_pseudogt_dqa_probe_2h/results/aggregation_summary.csv
server_calibration: /app/Object_Detection/efficient_teacher/efficientteacher_pseudogt_dqa_probe_2h/results/server_calibration_summary.csv
recommendation: /app/Object_Detection/efficient_teacher/efficientteacher_pseudogt_dqa_probe_2h/results/recommendation_table.csv
leaderboard: /app/Object_Detection/efficient_teacher/efficientteacher_pseudogt_dqa_probe_2h/results/overall_leaderboard.csv
